# Hybrid network-stress inference

Use the same class as A2's shared inference entry point to select a historical
prediction origin, predict next-hour stress, and return Prolog interpretations.
Requires a trained bundle, PySwip and SWI-Prolog. Recommendations are advisory.


In [ ]:
from pathlib import Path
import sys
import pandas as pd

A2_ROOT = next(candidate
               for parent in (Path.cwd(), *Path.cwd().parents)
               for candidate in (parent, parent / "A2_puja0009")
               if (candidate / "Source/models/NetworkStressModel.py").is_file())
SOURCE_DIR = A2_ROOT / "Source"
if str(SOURCE_DIR) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIR))
from config import TRANSFORMED_DATA_PATH, SAVED_MODELS_DIR
from models.NetworkStressModel import NetworkStressModel, prepare_modelling_data

# Generate with Source/A1_data_pipeline/run_pipeline.py, or set an existing transformed file.
DATA_PATH = TRANSFORMED_DATA_PATH
MODEL_DIRECTORY = SAVED_MODELS_DIR
data = pd.read_parquet(DATA_PATH)


## Select a prediction origin

Use `None` for the latest observation, or enter a timezone-aware historical time.
This version uses the latest branch's simplified rules; reserve-margin overrides
and symbolic risk escalation are not implemented.


In [ ]:
TIMESTAMP = None
model = NetworkStressModel(model_dir=MODEL_DIRECTORY)
from models.NetworkStressModel import normalise_timestamp, print_result

# Select and sort the history before passing its latest row to the model.
history = data.set_index("timestamp").sort_index()
if TIMESTAMP is not None:
    timestamp = normalise_timestamp(TIMESTAMP, pd.Series(history.index))
    if timestamp not in history.index:
        raise ValueError(f"No observation found for {timestamp}")
    history = history.loc[:timestamp]

result = model.inference(instance=history)
print_result(result)
display(pd.Series(result))
